# Chapter 05. 코사인 유사도로 비슷한 책 추천하기

# 실습 1. 필요한 라이브러리 확인하기

> **학습 목표**
> - 이번 Chapter에서 도서 추천 실습을 진행하는 데 필요한 핵심 라이브러리 목록을 확인합니다.
> - 가상환경 패키지 설치 명령어와 기본 `import` 구문을 파악합니다.

이번 Chapter에서 주로 사용하는 라이브러리는 다음과 같습니다.

* `pandas`
* `numpy`
* `scikit-learn`
* `scipy`
* `joblib`

필요한 패키지가 없다면 현재 가상환경에서 설치합니다.

```bash
python -m pip install pandas numpy scikit-learn scipy joblib

In [1]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

**실습 결과 및 설명**
- 코드 실행 시 아무런 에러 메시지가 출력되지 않고 정상 종료된다면 추천 실습을 위한 라이브러리 준비가 완료된 것입니다.
- 이번 실습에서도 처음부터 모든 코드를 한 셀에 작성하지 않고, 각 단계별로 결과를 확인하면서 진행합니다.

# 실습 2. Chapter 01 전처리 데이터 불러오기

> **학습 목표**
> - Chapter 01에서 전처리하여 저장한 베스트셀러 도서 데이터를 불러옵니다.
> - 데이터의 크기, 컬럼 목록, 결측치 및 중복 제목 존재 여부를 확인합니다.

이번 Chapter에서도 Chapter 01에서 만든 `book_bestseller_clean.csv` 파일을 사용합니다.

Notebook과 데이터 파일을 같은 폴더에 둔다고 가정합니다.

In [2]:
DATA_PATH = "book_bestseller_clean.csv"
df_books = pd.read_csv(DATA_PATH, encoding="utf-8-sig")

print("데이터 크기:", df_books.shape)
print("컬럼:", df_books.columns.tolist())
print("상품명 결측치:", df_books["상품명"].isna().sum())
print("상품명 중복:", df_books["상품명"].duplicated().sum())

df_books[["상품명"]].head(10)

데이터 크기: (199, 8)
컬럼: ['순위', '판매상품ID', '상품명', '판매가', '저자', '출판사', '발행일', '분야']
상품명 결측치: 0
상품명 중복: 0


,상품명
0,소년이 온다
1,모순
2,결국 국민이 합니다
3,혼모노
4,급류
5,초역 부처의 말
6,청춘의 독서(특별증보판)
7,어른의 행복은 조용하다
8,채식주의자
9,단 한 번의 삶(강물에디션 활판인쇄 한정판)


In [3]:
# 중복 제목이 있을 경우 실제 행 확인 (바로 삭제하지 않음)
duplicate_titles = df_books[
    df_books["상품명"].duplicated(keep=False)
].sort_values("상품명")

duplicate_titles.head(20)

,순위,판매상품ID,상품명,판매가,저자,출판사,발행일,분야


**실습 결과 및 설명**
- 파일이 정상적으로 열리고 한글 상품명이 깨지지 않고 출력되는지 확인합니다.
- 중복 제목이 존재하더라도 판형, 세트, 개정판, 상품 ID 등이 다를 수 있으므로 바로 삭제하지 않고 실제 데이터를 먼저 검토합니다.

# 실습 3. 추천에 사용할 데이터 준비하기

이번 추천에서 필수 컬럼은 **상품명** 입니다.
추천 결과를 보기 좋게 만들기 위해 다음 컬럼이 있으면 함께 사용합니다.

> 상품명
> 인물
> 출판사
> 분야
> 판매상품 ID

먼저 제목을 문자열로 정리합니다.

In [5]:
df_reco = df_books.copy()
df_reco["상품명"] = (
    df_reco["상품명"]
    .fillna("")
    .astype(str)
    .str.strip()
)

df_reco = df_reco[df_reco["상품명"] != ""].reset_index(drop=True)
print("추천에 사용할 도서 수:", len(df_reco))

추천에 사용할 도서 수: 199


`reset_index(drop=True)`를 사용하는 이유는 이후 TF-IDF 행렬의 행 번호와 DataFrame의 행 번호를 맞추기 위해서입니다.

> DataFrame index 0 ↔ TF-IDF matrix row 0
> DataFrame index 1 ↔ TF-IDF matrix row 1

이 대응 관계가 깨지면 추천 결과와 실제 도서 정보가 어긋날 수 있습니다.

# 실습 4. 콘텐츠 기반 추천이 무엇인지 이해하기

> **학습 목표**
> - 추천 시스템의 한 종류인 '콘텐츠 기반 추천'의 개념을 이해합니다.
> - 이번 실습에서 어떤 데이터를 사용하고, 어떤 데이터를 사용하지 않는지 명확히 구분합니다.

추천 시스템에는 여러 방식이 있습니다.
이번 실습에서는 **콘텐츠 기반 추천(Content-Based Recommendation)** 을 사용합니다.
초보자 단계에서는 다음처럼 이해하면 충분합니다.

> 선택한 항목의 특징
> ↓
> 비슷한 특징을 가진 다른 항목 찾기

이번에는 도서의 특징으로 제목 텍스트를 사용합니다.

> 도서 A 제목 → TF-IDF 벡터 A
> 도서 B 제목 → TF-IDF 벡터 B
> A와 B 벡터 비교 → 비슷하면 추천 후보

이번 추천은 다음 정보를 사용하지 않습니다.

* 사용자 구매 이력
* 사용자 클릭 이력
* 평점
* 찜 목록
* 판매량
* 독자 성별·연령
* 개인 취향

따라서 개인화 추천이라고 부르기보다는 **도서 제목 기반 유사 도서 추천** 이라고 표현하는 것이 정확합니다.

**실습 결과 및 설명**
- 이번 실습은 코드를 실행하는 대신, 앞으로 만들 추천 시스템의 성격을 정의하는 단계입니다.
- 우리가 만들 모델은 독자의 취향(Personalization)이 아닌, '책 제목의 텍스트 패턴'을 비교한다는 점을 꼭 기억해 주세요.

# 실습 5. 먼저 작은 벡터로 유사도를 생각해 보기

TF-IDF에 바로 들어가기 전에 아주 작은 숫자 벡터를 생각해 봅니다.

In [6]:
book_a = np.array([1, 1])
book_b = np.array([2, 2])
book_c = np.array([1, 0])

`book_a`와 `book_b`는 크기는 다르지만 같은 방향을 향합니다.

> A = [1, 1]
> B = [2, 2]

반면 `book_c`는 방향이 다릅니다.

> C = [1, 0]

코사인 유사도는 단순히 숫자 차이만 보는 것이 아니라 **벡터의 방향이 얼마나 비슷한지**를 봅니다.

**실습 결과 및 설명**
- 이 단계는 파이썬 코드 실행 결과가 따로 출력되지는 않습니다. 변수만 선언하는 과정입니다.
- 두 벡터가 가리키는 '방향'이 코사인 유사도의 핵심이라는 점을 직관적으로 이해하는 것이 이번 실습의 목표입니다.

# 실습 6. 코사인 유사도 개념 이해하기

두 벡터 *A*, *B*의 코사인 유사도는 다음과 같이 표현할 수 있습니다.

> cosine similarity(A, B) = (A · B) / (||A|| × ||B||)

여기서 중요한 것은 공식을 외우는 것이 아닙니다.
초보자 단계에서는 다음처럼 이해합니다.

> 방향이 매우 비슷함 → 1에 가까움
> 공통 방향이 거의 없음 → 0에 가까움

TF-IDF 벡터는 음수가 아닌 값으로 구성되므로 이번 실습에서 계산되는 유사도는 일반적으로 **0**에서 **1** 사이에서 해석할 수 있습니다.

**직접 계산해 보기**

In [7]:
from sklearn.metrics.pairwise import cosine_similarity

vectors = np.array([
    [1, 1],
    [2, 2],
    [1, 0],
])

similarities = cosine_similarity(vectors)
similarities

array([[1.        , 1.        , 0.70710678],
       [1.        , 1.        , 0.70710678],
       [0.70710678, 0.70710678, 1.        ]])

**실습 결과 및 설명**
- 행과 열은 각각 세 벡터를 의미합니다. 
- 출력된 3x3 배열을 보면, 자기 자신과의 유사도(대각선 값)는 1에 가깝게 나옵니다.
- 첫 번째 벡터 `[1, 1]`과 두 번째 벡터 `[2, 2]`의 유사도 역시 방향이 같으므로 1.0이 출력되는 것을 확인할 수 있습니다.

# 실습 7. 왜 자기 자신과의 유사도가 가장 높은가?

어떤 벡터를 자기 자신과 비교하면 방향이 완전히 같습니다.
따라서 일반적으로 다음과 같은 결과가 나옵니다.

> book A vs book A → 1.0
> book A vs book B → 0.73
> book A vs book C → 0.21

추천을 만들 때 아무 처리도 하지 않으면 선택한 도서 자기 자신이 1위가 됩니다.
하지만 추천 목적은 *다른 도서* 를 찾는 것입니다.
그래서 나중에 반드시 자기 자신을 제외합니다.

**실습 결과 및 설명**
- 이번 실습은 별도의 파이썬 코드 실행 없이, '왜 코사인 유사도를 구한 후 자기 자신을 제외해야 하는지' 그 당위성을 이해하는 과정입니다.
- 이후 추천 시스템을 구현할 때, 가장 높은 점수(1.0)를 가진 1위를 제외하고 2위부터 추천 목록에 넣는 로직이 필요하다는 점을 기억해 주세요.

# 실습 8. 작은 문장으로 TF-IDF와 유사도 연결하기

> **학습 목표**
> - 간단한 3개 문장 예제를 통해 TF-IDF 벡터 변환과 코사인 유사도 계산이 어떻게 동작하는지 직관적으로 확인합니다.
> - 공통 단어 유무에 따른 유사도 점수의 차이를 이해합니다.

실제 데이터 전에 작은 문장을 이용합니다.

In [8]:
sample_titles = [
    "파이썬 데이터 분석",
    "파이썬 머신러닝 입문",
    "영어 회화 기초",
]

In [9]:
# TF-IDF 벡터를 만듭니다.
sample_vectorizer = TfidfVectorizer()
sample_matrix = sample_vectorizer.fit_transform(sample_titles)
print("행렬 크기:", sample_matrix.shape)
print("단어 목록:", sample_vectorizer.get_feature_names_out())

행렬 크기: (3, 8)
단어 목록: ['기초' '데이터' '머신러닝' '분석' '영어' '입문' '파이썬' '회화']


In [10]:
# 이제 첫 번째 제목과 전체 제목을 비교합니다.
sample_scores = cosine_similarity(
    sample_matrix[0],
    sample_matrix
).flatten()
sample_scores

array([1.      , 0.224325, 0.      ])

**실습 결과 및 설명**
예상되는 관계는 다음과 같습니다.

> 파이썬 데이터 분석 ↔ 파이썬 데이터 분석 (가장 높음 / 1.0)
> 파이썬 데이터 분석 ↔ 파이썬 머신러닝 입문 ('파이썬'이라는 공통 단어가 있음)
> 파이썬 데이터 분석 ↔ 영어 회화 기초 (공통 단어가 거의 없음 / 0.0)

- 첫 번째 문장과 자기 자신을 비교하면 유사도가 1.0이 됩니다.
- '파이썬'이라는 공통 단어가 들어간 두 번째 문장과는 양의 유사도 값을 가지며, 공통 단어가 완전히 없는 세 번째 문장과의 유사도는 0이 됩니다.
- 정확한 숫자는 실제 코드를 실행하여 출력 결과를 확인합니다.

# 실습 9. 실제 도서 제목을 TF-IDF로 변환하기

> **학습 목표**
> - 전처리된 실제 도서 상품명을 TfidfVectorizer를 통해 수치형 행렬(TF-IDF Matrix)로 변환합니다.
> - 변환된 TF-IDF 행렬의 크기(도서 수, 단어 수)를 확인하고 행과 열의 의미를 이해합니다.

이제 실제 *상품명* 을 사용합니다.

행은 도서이고, 열은 TF-IDF 단어 사전의 단어입니다.

> 행 = 도서  
> 열 = 단어  
> 값 = TF-IDF 가중치  

이번 추천 실습에서는 현재 카탈로그 안에서 도서 간 유사도를 비교하므로 전체 추천 대상 제목으로 TF-IDF 표현을 만드는 방식으로 진행합니다.
Chapter 04의 train/test 분류와 목적이 다르다는 점을 구분합니다.

In [11]:
# 1. 추천 대상 도서의 상품명 컬럼 가져오기
titles = df_reco["상품명"]

# 2. TF-IDF Vectorizer 객체 생성
tfidf = TfidfVectorizer()

# 3. 도서 제목 텍스트 데이터를 TF-IDF 행렬로 변환 (fit_transform)
tfidf_matrix = tfidf.fit_transform(titles)

# 4. 변환된 TF-IDF 행렬의 크기(도서 수, 단어 수) 확인
print("도서 수:", tfidf_matrix.shape[0])
print("단어 수:", tfidf_matrix.shape[1])

도서 수: 199
단어 수: 536


**실습 결과 및 설명**
- `df_reco["상품명"]` 데이터셋 전체에 대해 TF-IDF 벡터 변환이 정상적으로 수행되었습니다.
- 행렬의 행(Row) 수는 전체 도서 개수와 일치하고, 열(Column) 수는 추출된 단어 사전의 총 단어 개수를 나타냅니다.

# 실습 10. 추천에서는 왜 전체 카탈로그에 TF-IDF를 fit할 수 있을까?

> **학습 목표**
> - 지도학습(분류) 문제와 추천 시스템(유사도 계산) 문제에서 TF-IDF `fit_transform` 범위의 차이점을 이해합니다.
> - 데이터 누수(Data Leakage)의 개념과 추천 카탈로그 전처리 방식을 파악합니다.

Chapter 04에서는 다음 원칙을 배웠습니다.

> 분류 성능 평가  
> → train에만 Vectorizer fit  
> → test는 transform만 수행  

테스트 데이터의 정보를 미리 보면 **데이터 누수(Data Leakage)**가 발생할 수 있기 때문입니다.

하지만 이번 Chapter의 기본 실습은 모델 성능을 train/test로 평가하는 분류 문제가 아닙니다.  
현재 보유한 도서 카탈로그 전체 안에서 서로 비슷한 항목을 찾는 것이 목적입니다.

따라서 다음처럼 전체 카탈로그 제목을 이용해 추천용 표현을 만들어 사용할 수 있습니다.

In [12]:
# 추천 시스템(카탈로그 전체 유사도 검색)에서는 
# 전체 도서 제목 데이터에 대해 TF-IDF를 fit_transform하여 행렬을 생성합니다.
tfidf_matrix = tfidf.fit_transform(df_reco["상품명"])

# 생성된 TF-IDF 행렬의 크기(전체 도서 수, 전체 단어 수) 확인
print("전체 도서 수:", tfidf_matrix.shape[0])
print("전체 단어 수:", tfidf_matrix.shape[1])

전체 도서 수: 199
전체 단어 수: 536


**실습 결과 및 설명**
- 분류 모델과 달리 카탈로그 내부 검색/추천 시스템에서는 전체 데이터셋을 기반으로 단어 사전을 구축하는 것이 일반적입니다.
- 다만, 추천 알고리즘을 별도의 평가 데이터로 엄밀하게 평가하는 실험을 설계한다면 그 평가 목적에 맞는 데이터 분리와 누수 방지 전략을 다시 정의해야 합니다.
- 이번 Chapter에서는 **현재 카탈로그 내부 유사도 추천**에 집중합니다.

# 실습 11. TF-IDF 단어 사전 확인하기

> **학습 목표**
> - `TfidfVectorizer`를 통해 추출된 단어 사전(Feature Names)을 확인합니다.
> - 단어 목록의 특성(한글, 영문, 숫자 등)을 점검하여 전처리 상태를 검토하는 방법을 이해합니다.

TF-IDF 단어 사전을 확인합니다.

In [13]:
# 1. TF-IDF 단어 사전(Feature Names) 추출
feature_names = tfidf.get_feature_names_out()

# 2. 전체 단어 수 및 상위 50개 단어 출력하여 확인
print("전체 단어 수:", len(feature_names))
print(feature_names[:50])

전체 단어 수: 536
['100' '100만' '100일' '10만' '10일' '10주년' '110' '13' '14' '19' '1984' '1차'
 '2025' '2026' '20만' '20주년' '28시간에' '29' '30만' '30만부' '3개의' '400쇄' '40대에'
 '50만' '50만부' '66가지' '700' '70만' '750' '850' 'advanced' 'ai' 'all' 'etf'
 'etf로' 'ets' 'hackers' 'in' 'lc' 'listening' 'one' 'opic' 'ox' 'rc'
 'reading' 'steal' 'stop' 'the' 'voca' 'vocabulary']


단어 목록을 보고 다음을 확인합니다.

* 한글 단어가 정상적으로 들어갔는가?
* 영문이 어떻게 처리되었는가?
* 숫자가 포함되어 있는가?
* 의미가 적은 단어가 너무 많은가?

**실습 결과 및 설명**
- 추천 결과가 이상하게 나올 때는 추천 알고리즘 코드만 볼 것이 아니라, **TF-IDF에 어떤 단어가 들어갔는지**를 먼저 확인해야 합니다.
- 추출된 50개 단어 목록을 통해 텍스트 토큰화가 정상적으로 완료되었는지 확인합니다.

# 실습 12. 한 권의 도서를 선택하기

> **학습 목표**
> - 기준이 되는 도서를 index 번호로 선택하고, 해당 index의 실제 상품명을 확인합니다.
> - index 번호와 실제 도서 제목을 연결하여 어떤 책을 기준으로 유사도를 계산하는지 정확히 파악합니다.

먼저 index 번호로 한 권을 선택해 봅니다.

index만 보고 추천하면 어떤 책을 기준으로 계산했는지 놓치기 쉽습니다.

In [14]:
# 1. 추천 기준이 될 도서의 index 번호 지정 (예: 0번)
selected_index = 0

# 2. 지정한 index의 실제 상품명(도서 제목) 출력하여 확인
print("선택한 도서:", df_reco.loc[selected_index, "상품명"])

선택한 도서: 소년이 온다


**실습 결과 및 설명**
- 지정한 index 위치의 실제 도서 제목이 정상적으로 출력되는지 확인합니다.
- index 번호만 사용해 추천을 수행하면 어떤 도서를 기준으로 계산했는지 파악하기 어려우므로, 항상 실제 상품명을 함께 확인하고 진행합니다.

# 실습 13. 선택한 도서와 전체 도서의 유사도 계산하기

> **학습 목표**
> - 선택한 도서의 TF-IDF 벡터와 전체 도서 목록의 TF-IDF 행렬 간 코사인 유사도를 계산합니다.
> - 계산된 유사도 점수 배열의 길이와 DataFrame 행 개수가 일치하는 대응 관계를 확인합니다.

선택한 도서의 TF-IDF 벡터를 추출하고, 전체 도서 벡터와 비교하여 코사인 유사도를 계산합니다.

In [15]:
# 1. 선택한 도서의 TF-IDF 벡터 가져오기
selected_vector = tfidf_matrix[selected_index]

# 2. 선택한 도서 벡터와 전체 도서 TF-IDF 행렬 간 코사인 유사도 계산
#    flatten()을 사용하여 2차원 배열 결과(1, N)를 1차원 배열(N,)로 변환합니다.
similarity_scores = cosine_similarity(
    selected_vector,
    tfidf_matrix
).flatten()

# 3. 유사도 점수 배열의 길이와 전체 도서 수(DataFrame 행 개수) 비교 확인
print("유사도 점수 개수:", len(similarity_scores))
print("전체 도서 수:", len(df_reco))

유사도 점수 개수: 199
전체 도서 수: 199


**실습 결과 및 설명**
- `len(similarity_scores)`와 `len(df_reco)` 두 값이 동일하게 출력되는지 확인합니다.
- 각 점수는 DataFrame의 행과 다음과 같은 1:1 대응 관계를 가집니다.

> `similarity_scores[0]` ↔ `df_reco.iloc[0]`  
> `similarity_scores[1]` ↔ `df_reco.iloc[1]`  

- 이 대응 관계가 유지되는 것이 도서 추천 결과와 실제 메타데이터를 매칭하는 핵심 원리입니다.

# 실습 14. 유사도 점수 일부 확인하기

> **학습 목표**
> - 계산된 유사도 점수 배열의 일부를 확인해 봅니다.
> - 단순 수치 데이터만으로는 어떤 도서인지 식별하기 어려우므로, DataFrame으로 결합하여 도서 제목과 유사도 점수를 한눈에 매칭하는 방법을 이해합니다.

유사도 점수 배열을 그대로 출력하면 단순 수치 목록만 보여 어떤 도서에 대한 점수인지 알기 어렵습니다.
따라서 DataFrame을 새로 만들어 도서 index, 상품명, 그리고 계산된 코사인 유사도 점수를 연결하여 함께 확인합니다.

In [16]:
# 1. 유사도 점수 배열의 앞선 10개 값 직접 출력해보기
print("유사도 점수 배열 일부:", similarity_scores[:10])

# 2. 도서 index, 상품명, 유사도 점수를 한눈에 볼 수 있는 DataFrame 생성
score_df = pd.DataFrame({
    "index": np.arange(len(df_reco)),
    "상품명": df_reco["상품명"],
    "similarity": similarity_scores,
})

# 3. 도서 제목과 유사도 점수가 매칭된 상위 10개 행 출력
score_df.head(10)

유사도 점수 배열 일부: [1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


,index,상품명,similarity
0,0,소년이 온다,1.0
1,1,모순,0.0
2,2,결국 국민이 합니다,0.0
3,3,혼모노,0.0
4,4,급류,0.0
5,5,초역 부처의 말,0.0
6,6,청춘의 독서(특별증보판),0.0
7,7,어른의 행복은 조용하다,0.0
8,8,채식주의자,0.0
9,9,단 한 번의 삶(강물에디션 활판인쇄 한정판),0.0


**실습 결과 및 설명**
- 수치 배열로만 존재하던 점수를 `df_reco`의 상품명 컬럼과 연결하여 DataFrame 형태로 구성했습니다.
- 이제 선택한 도서(index 0)와 다른 도서들 간의 유사도가 얼마인지 제목과 함께 명확히 확인할 수 있습니다.
- 선택한 기준 도서 자신과의 유사도(0번 index 행)는 가장 높은 점수인 `1.0`으로 출력되는 것을 볼 수 있습니다.

# 실습 15. 가장 높은 유사도 찾기

> **학습 목표**
> - 유사도 점수를 기준으로 내림차순 정렬을 수행하여 상위 도서 목록을 확인합니다.
> - 정렬 결과 최상단(1위)에 선택한 자기 자신이 위치하는 구조를 이해합니다.

생성한 `score_df`를 유사도 점수 기준으로 내림차순 정렬하여 어떤 도서가 상위에 올라오는지 확인합니다.

In [17]:
# 1. similarity(유사도 점수) 컬럼을 기준으로 내림차순(ascending=False) 정렬
# 2. 상위 10개 도서 항목 출력하여 확인
score_df.sort_values(
    "similarity",
    ascending=False
).head(10)

,index,상품명,similarity
0,0,소년이 온다,1.0
1,1,모순,0.0
2,2,결국 국민이 합니다,0.0
3,3,혼모노,0.0
4,4,급류,0.0
5,5,초역 부처의 말,0.0
6,6,청춘의 독서(특별증보판),0.0
7,7,어른의 행복은 조용하다,0.0
8,8,채식주의자,0.0
9,9,단 한 번의 삶(강물에디션 활판인쇄 한정판),0.0


**실습 결과 및 설명**
- 가장 위(1위)에는 일반적으로 선택한 자기 자신 도서가 위치합니다.
- 자기 자신과의 비교이므로 유사도는 `1.0`으로 나타나며 이는 지극히 정상적인 현상입니다.
- 다음 실습에서는 추천 목록에서 1위인 자기 자신을 제외하는 로직을 작성합니다.